# ARIAN Phase 3 — Weather Prediction & Visualization

Predicts **every** available weather measurement **hourly** over a continuous
**30-day** horizon using a Prophet × SARIMA × XGBoost stacking ensemble,
trained and forecast independently per city.

Produces an interactive Folium + Plotly map where the user can select a date
to view predicted weather across all 16 cities.

**Inputs:**
  - `data/raw/weather_hourly.parquet` — hourly historical weather (from NB 1)
  - `data/reference/cities.parquet`   — city coordinates

**Outputs:**
  - `outputs/phase3_weather_hourly_30d.parquet` — full hourly 30-day forecast
  - `outputs/phase3_weather_leaderboard.csv`    — model leaderboard
  - `outputs/phase3_weather_map.html`           — interactive date-selectable map

## §0 · Setup

In [24]:
%pip install -q prophet statsmodels xgboost scikit-learn pandas numpy \
    pyarrow tqdm plotly folium holidays scipy 2>/dev/null || \
!pip install -q prophet statsmodels xgboost scikit-learn pandas numpy \
    pyarrow tqdm plotly folium holidays scipy

Note: you may need to restart the kernel to use updated packages.


In [25]:
# ─── ARIAN — Universal environment & path setup ───────────────────────────
import os, sys
from pathlib import Path

PROJECT_NAME = "ARIAN_Data"

def _detect_project_root() -> Path:
    if os.environ.get("ARIAN_ROOT"):
        return Path(os.environ["ARIAN_ROOT"]).expanduser().resolve()
    if "google.colab" in sys.modules:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        return Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
        if cand.name == PROJECT_NAME and (cand / "data").is_dir():
            return cand
    return here.parent if here.name == "notebooks" else here

ROOT      = _detect_project_root()
DATA      = ROOT / "data"
RAW       = DATA / "raw"
LEGACY    = RAW  / "legacy"
PROCESSED = DATA / "processed"
REFERENCE = DATA / "reference"
OUTPUTS   = ROOT / "outputs"
MODELS    = ROOT / "models"

for p in (RAW, LEGACY, PROCESSED, REFERENCE, OUTPUTS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root : {ROOT}")

📁 Project root : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF


In [26]:
# ─── Imports ───────────────────────────────────────────────────────────────
import warnings, logging, json
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

from datetime import datetime, timedelta, timezone
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.optimize import minimize

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import holidays
import folium
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load hourly data ───────────────────────────────────────────────────────
HOURLY_PATH = RAW / "weather_hourly.parquet"
if not HOURLY_PATH.exists():
    raise FileNotFoundError(f"Missing {HOURLY_PATH}. Run notebook 01 first.")

hourly = pd.read_parquet(HOURLY_PATH)
hourly["Timestamp"] = pd.to_datetime(hourly["Timestamp"])
hourly = hourly.sort_values(["City", "Timestamp"]).reset_index(drop=True)

# Load city coordinates
cities_path = REFERENCE / "cities.parquet"
if cities_path.exists():
    cities_df = pd.read_parquet(cities_path)
    CITIES = {r["City"]: (r["Latitude"], r["Longitude"]) for _, r in cities_df.iterrows()}
else:
    CITIES = {c: (hourly.loc[hourly["City"] == c, "Latitude"].iloc[0],
                   hourly.loc[hourly["City"] == c, "Longitude"].iloc[0])
              for c in hourly["City"].unique()}

# All numeric weather columns (excluding City, Lat, Lon, Timestamp)
WEATHER_FEATURES = [c for c in hourly.columns
                    if c not in ("City", "Latitude", "Longitude", "Timestamp")
                    and hourly[c].dtype in ("float64", "float32", "int64")]

print(f"Loaded {len(hourly):,} hourly rows · {hourly['City'].nunique()} cities")
print(f"Date range: {hourly['Timestamp'].min()} → {hourly['Timestamp'].max()}")
print(f"Weather features to predict ({len(WEATHER_FEATURES)}): {WEATHER_FEATURES}")

# ── Data completeness audit ────────────────────────────────────────────────
print("\n📊 Feature completeness:")
n_ok, n_bad = 0, 0
for feat in WEATHER_FEATURES:
    nn = hourly[feat].notna().sum()
    pct = nn / len(hourly) * 100
    if pct < 1:
        print(f"   ⚠️  {feat:30s}  {pct:.1f}% data — CRITICAL: re-run NB1 to fetch!")
        n_bad += 1
    else:
        print(f"   ✅ {feat:30s}  {pct:.1f}% data")
        n_ok += 1
if n_bad > 0:
    print(f"\n⚠️  {n_bad} features have NO data. Re-run 01_Data_Ingestion.ipynb first!")
else:
    print(f"\n✅ All {n_ok} features have data.")

Loaded 2,001,408 hourly rows · 16 cities
Date range: 2012-01-19 20:00:00 → 2026-04-27 19:00:00
Weather features to predict (9): ['Temperature_C', 'Humidity_percent', 'Rain_mm', 'Wind_Speed_kmh', 'Wind_Direction_deg', 'Pressure_hPa', 'Solar_Radiation_Wm2', 'Soil_Temp_C', 'Soil_Moisture']

📊 Feature completeness:
   ✅ Temperature_C                   100.0% data
   ✅ Humidity_percent                100.0% data
   ✅ Rain_mm                         100.0% data
   ✅ Wind_Speed_kmh                  100.0% data
   ⚠️  Wind_Direction_deg              0.0% data — CRITICAL: re-run NB1 to fetch!
   ⚠️  Pressure_hPa                    0.0% data — CRITICAL: re-run NB1 to fetch!
   ⚠️  Solar_Radiation_Wm2             0.0% data — CRITICAL: re-run NB1 to fetch!
   ⚠️  Soil_Temp_C                     0.0% data — CRITICAL: re-run NB1 to fetch!
   ⚠️  Soil_Moisture                   0.0% data — CRITICAL: re-run NB1 to fetch!

⚠️  5 features have NO data. Re-run 01_Data_Ingestion.ipynb first!


## §1 · Configuration

In [27]:
FORECAST_HOURS = 30 * 24   # 30 days x 24 hours = 720 hours
TEST_HOURS     = 7 * 24    # hold out last 7 days for evaluation
VAL_HOURS      = 7 * 24    # 7-day validation for stacking weights (was 3 days)
MIN_TRAIN_HOURS = 365 * 24 # require >= 1 year of history

# XGBoost hyperparameters — accuracy-focused
XGB_PARAMS = dict(
    n_estimators=800, learning_rate=0.03, max_depth=7,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.1, tree_method="hist",
    n_jobs=-1, random_state=42,
)
XGB_EARLY_STOP = 50  # early stopping rounds on validation split

# SARIMA order — hourly data with daily seasonality (period=24)
SARIMA_ORDER          = (1, 0, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 1, 24)

# Azerbaijan holidays for Prophet
AZ_HOLIDAYS = pd.DataFrame(
    [{"holiday": "Novruz",       "ds": pd.Timestamp(y, 3, 21)} for y in range(2012, 2030)] +
    [{"holiday": "Independence", "ds": pd.Timestamp(y, 10, 18)} for y in range(2012, 2030)] +
    [{"holiday": "Republic",     "ds": pd.Timestamp(y, 5, 28)} for y in range(2012, 2030)]
)
AZ_HOLIDAYS["lower_window"], AZ_HOLIDAYS["upper_window"] = -1, 2

# Downsample factor for SARIMA (fit on 6-hourly to stay tractable)
SARIMA_DOWNSAMPLE = 6

# Physical bounds for each weather variable (min, max)
PHYSICAL_BOUNDS = {
    "Temperature_C":      (-40, 55),
    "Humidity_percent":    (0, 100),
    "Rain_mm":             (0, 100),
    "Wind_Speed_kmh":      (0, 200),
    "Wind_Direction_deg":  (0, 360),
    "Pressure_hPa":        (850, 1080),
    "Solar_Radiation_Wm2": (0, 1400),
    "Soil_Temp_C":         (-30, 60),
    "Soil_Moisture":       (0, 1),
    "Dew_Point_C":         (-60, 40),
    "VPD_kPa":             (0, 8),
}

# Units for display
FEAT_UNITS = {
    "Temperature_C": "C", "Humidity_percent": "%", "Rain_mm": "mm",
    "Wind_Speed_kmh": "km/h", "Wind_Direction_deg": "deg",
    "Pressure_hPa": "hPa", "Solar_Radiation_Wm2": "W/m2",
    "Soil_Temp_C": "C", "Soil_Moisture": "",
    "Dew_Point_C": "C", "VPD_kPa": "kPa",
}

# Features that benefit from log-transform (right-skewed, non-negative)
LOG_FEATURES = {"Rain_mm", "Solar_Radiation_Wm2", "Wind_Speed_kmh"}

# Features where multiplicative Prophet seasonality is better
MULTIPLICATIVE_FEATURES = {"Rain_mm", "Solar_Radiation_Wm2", "Wind_Speed_kmh"}

print(f"Forecast horizon: {FORECAST_HOURS} hours ({FORECAST_HOURS // 24} days)")
print(f"Test holdout:     {TEST_HOURS} hours ({TEST_HOURS // 24} days)")
print(f"Validation:       {VAL_HOURS} hours ({VAL_HOURS // 24} days)")
print(f"Features to predict: {len(WEATHER_FEATURES)}")
print(f"Cities: {len(CITIES)}")
print(f"XGBoost: {XGB_PARAMS['n_estimators']} trees, early_stop={XGB_EARLY_STOP}")

Forecast horizon: 720 hours (30 days)
Test holdout:     168 hours (7 days)
Validation:       168 hours (7 days)
Features to predict: 9
Cities: 16
XGBoost: 800 trees, early_stop=50


## §2 · Feature engineering helpers for XGBoost

In [28]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Calendar / cyclical features for hourly data."""
    df = df.copy()
    ts = df["Timestamp"]
    df["hour"]      = ts.dt.hour
    df["dayofweek"] = ts.dt.dayofweek
    df["dayofyear"] = ts.dt.dayofyear
    df["month"]     = ts.dt.month
    df["week"]      = ts.dt.isocalendar().week.astype(int)
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["doy_sin"]   = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["doy_cos"]   = np.cos(2 * np.pi * df["dayofyear"] / 365.25)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_lag_features_hourly(df: pd.DataFrame, target: str,
                            lags=(1, 2, 3, 6, 12, 24, 48, 168, 336),
                            rolls=(24, 72, 168)) -> pd.DataFrame:
    """Per-city lags and rolling stats for hourly series."""
    df = df.copy()
    s = df[target]
    for L in lags:
        df[f"lag_{L}"] = s.shift(L)
    for W in rolls:
        rolled = s.shift(1).rolling(W, min_periods=1)
        df[f"roll_{W}_mean"] = rolled.mean()
        df[f"roll_{W}_std"]  = rolled.std()
        df[f"roll_{W}_min"]  = rolled.min()
        df[f"roll_{W}_max"]  = rolled.max()
    # Trend: difference between short and long rolling means
    if "roll_24_mean" in df.columns and "roll_168_mean" in df.columns:
        df["trend_24_168"] = df["roll_24_mean"] - df["roll_168_mean"]
    return df


TIME_FEAT_COLS = ["hour", "dayofweek", "dayofyear", "month", "week",
                  "hour_sin", "hour_cos", "doy_sin", "doy_cos",
                  "month_sin", "month_cos"]


# ── Derived weather features ──────────────────────────────────────────────

def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add physically meaningful derived weather features."""
    df = df.copy()
    T = df.get("Temperature_C")
    H = df.get("Humidity_percent")

    if T is not None and H is not None:
        H_safe = H.clip(1, 100) / 100.0
        # Dew point (Magnus formula)
        a, b = 17.27, 237.7
        gamma = a * T / (b + T) + np.log(H_safe)
        df["Dew_Point_C"] = b * gamma / (a - gamma)

        # Saturation vapor pressure (Tetens)
        e_sat = 0.6108 * np.exp(a * T / (b + T))
        e_act = e_sat * H_safe
        df["VPD_kPa"] = (e_sat - e_act).clip(lower=0)  # Vapor pressure deficit

    return df


def add_cross_feature_context(df: pd.DataFrame,
                              all_features: list,
                              target: str) -> pd.DataFrame:
    """Add rolling means of OTHER weather features as cross-variable context.
    These give XGBoost access to inter-variable correlations."""
    df = df.copy()
    cross_feats = [f for f in all_features if f != target and f in df.columns]
    for feat in cross_feats:
        s = df[feat]
        if s.notna().sum() < 48:
            continue
        df[f"x_{feat}_24h"] = s.shift(1).rolling(24, min_periods=1).mean()
        df[f"x_{feat}_168h"] = s.shift(1).rolling(168, min_periods=1).mean()
    return df


# ── Seasonal climatology baseline ─────────────────────────────────────────

def compute_seasonal_naive(city_df: pd.DataFrame, feat: str,
                           future_timestamps, history_years: int = 3) -> np.ndarray:
    """Climatological baseline: mean of same (month, day, hour) from last N years."""
    df = city_df[["Timestamp", feat]].dropna(subset=[feat]).copy()
    max_year = int(df["Timestamp"].dt.year.max())
    recent = df[df["Timestamp"].dt.year >= max_year - history_years].copy()

    recent["mdh"] = (recent["Timestamp"].dt.month * 10000 +
                     recent["Timestamp"].dt.day   * 100   +
                     recent["Timestamp"].dt.hour)
    clim = recent.groupby("mdh")[feat].mean()

    ft = pd.DatetimeIndex(future_timestamps)
    if ft.tz is not None:
        ft = ft.tz_localize(None)
    ft_mdh = ft.month * 10000 + ft.day * 100 + ft.hour
    out = pd.Series(ft_mdh).map(clim).values.astype(float)

    monthly = recent.groupby(recent["Timestamp"].dt.month)[feat].mean()
    for i in range(len(out)):
        if np.isnan(out[i]):
            out[i] = monthly.get(ft[i].month, np.nan)
    overall = float(city_df[feat].mean())
    out = np.where(np.isnan(out), overall, out)
    return out


# ── Physical bounds clipping ──────────────────────────────────────────────

def clip_to_bounds(values: np.ndarray, feat: str) -> np.ndarray:
    """Clip predictions to physically plausible ranges."""
    lo, hi = PHYSICAL_BOUNDS.get(feat, (-np.inf, np.inf))
    return np.clip(values, lo, hi)


# ── Horizon-aware ensemble blending ───────────────────────────────────────

def apply_horizon_blend(prophet_fc, sarima_fc, xgb_fc, seasonal_fc,
                        weights, n_hours):
    """Horizon-aware blending: XGBoost weight decays after 72 h,
    redistributed to Prophet and seasonal climatology for long-range
    stability."""
    hours = np.arange(n_hours, dtype=float)
    DECAY_START = 72.0
    decay = np.where(hours > DECAY_START,
                     np.exp(-0.012 * (hours - DECAY_START)), 1.0)

    w_p = weights["prophet"]
    w_s = weights["sarima"]
    w_x = weights["xgboost"]
    w_c = weights.get("seasonal", 0.0)

    xgb_w     = w_x * decay
    reduction = w_x - xgb_w
    prophet_w  = w_p + reduction * 0.50
    seasonal_w = w_c + reduction * 0.30
    sarima_w   = w_s + reduction * 0.20

    return (prophet_w  * prophet_fc  +
            sarima_w   * sarima_fc   +
            xgb_w      * xgb_fc      +
            seasonal_w * seasonal_fc)


print("Feature helpers ready  [+derived feats, +cross-feature context, "
      "+seasonal naive, +physical clipping, +horizon blend]")

Feature helpers ready  [+derived feats, +cross-feature context, +seasonal naive, +physical clipping, +horizon blend]


## §3 · Prophet trainer (per city × feature)

In [29]:
def fit_prophet_hourly(train_ts: pd.Series, train_idx: pd.DatetimeIndex,
                       target: str) -> Prophet:
    df = pd.DataFrame({"ds": train_idx, "y": train_ts.values})
    use_log = target in LOG_FEATURES
    if use_log:
        df["y"] = np.log1p(df["y"].clip(lower=0))

    # Per-feature tuning
    seasonality_mode = ("multiplicative" if target in MULTIPLICATIVE_FEATURES
                        else "additive")
    cps = 0.1 if target in ("Temperature_C", "Soil_Temp_C", "Pressure_hPa") else 0.05

    m = Prophet(
        yearly_seasonality=20,    # higher Fourier order for annual cycle
        weekly_seasonality=True,
        daily_seasonality=14,     # higher Fourier order for diurnal cycle
        holidays=AZ_HOLIDAYS,
        seasonality_mode=seasonality_mode,
        changepoint_prior_scale=cps,
        seasonality_prior_scale=10.0,
        n_changepoints=30,
        interval_width=0.80,
    )
    m.fit(df)
    m._arian_log = use_log
    return m


def predict_prophet_hourly(model: Prophet,
                           future_dates: pd.DatetimeIndex) -> np.ndarray:
    fc = model.predict(pd.DataFrame({"ds": future_dates}))
    yhat = fc["yhat"].values
    if getattr(model, "_arian_log", False):
        yhat = np.expm1(yhat).clip(min=0)
    return yhat

print("Prophet helpers ready  [per-feature seasonality tuning]")

Prophet helpers ready  [per-feature seasonality tuning]


## §4 · SARIMA trainer (per city × feature, downsampled for tractability)

In [30]:
def fit_sarima_hourly(train_ts: pd.Series, target: str):
    """Fit SARIMA on downsampled (every SARIMA_DOWNSAMPLE hours) series.
    Use the last 180 days for fitting — longer context for accuracy."""
    y = train_ts.values.copy()
    use_log = target in LOG_FEATURES
    if use_log:
        y = np.log1p(np.clip(y, 0, None))

    # Downsample
    y_ds = y[::SARIMA_DOWNSAMPLE]
    # Use last 180 days (was 90)
    max_len = (180 * 24) // SARIMA_DOWNSAMPLE
    if len(y_ds) > max_len:
        y_ds = y_ds[-max_len:]

    seasonal_period = 24 // SARIMA_DOWNSAMPLE  # daily cycle in downsampled steps
    try:
        fit = SARIMAX(
            y_ds,
            order=SARIMA_ORDER,
            seasonal_order=(1, 1, 1, max(seasonal_period, 2)),
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=100)
    except Exception:
        # Fallback to simpler model
        try:
            fit = SARIMAX(
                y_ds, order=(2, 0, 1),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False, maxiter=50)
        except Exception:
            fit = SARIMAX(
                y_ds, order=(1, 0, 0),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False, maxiter=30)

    fit._arian_log = use_log
    fit._arian_ds = SARIMA_DOWNSAMPLE
    return fit


def predict_sarima_hourly(fit, steps_hourly: int) -> np.ndarray:
    """Forecast downsampled steps then interpolate back to hourly."""
    ds = getattr(fit, "_arian_ds", SARIMA_DOWNSAMPLE)
    ds_steps = (steps_hourly + ds - 1) // ds
    yhat_ds = np.asarray(fit.forecast(steps=ds_steps))
    if getattr(fit, "_arian_log", False):
        yhat_ds = np.expm1(yhat_ds).clip(min=0)
    # Linear interpolation back to hourly
    x_ds = np.arange(len(yhat_ds)) * ds
    x_hr = np.arange(steps_hourly)
    yhat = np.interp(x_hr, x_ds, yhat_ds)
    return yhat

print("SARIMA helpers ready  [180-day context, maxiter=100]")

SARIMA helpers ready  [180-day context, maxiter=100]


## §5 · XGBoost trainer (per city × feature)

In [31]:
def get_xgb_feature_cols(df: pd.DataFrame) -> List[str]:
    """Return all numeric lag + time + cross-feature columns."""
    return [c for c in df.columns
            if c.startswith(("lag_", "roll_", "x_", "trend_")) or c in TIME_FEAT_COLS]


# ── Lag / rolling config (must match add_lag_features_hourly) ──────────────
_XGB_LAGS  = (1, 2, 3, 6, 12, 24, 48, 168, 336)
_XGB_ROLLS = (24, 72, 168)
_MAX_LOOKBACK = max(max(_XGB_LAGS), max(_XGB_ROLLS)) + 1   # 337


def fit_xgb_hourly(city_df: pd.DataFrame, target: str,
                    all_features: list = None):
    """Fit XGBoost with lag, rolling, time, AND cross-feature context.
    Uses early stopping on a chronological validation split for accuracy."""
    feat_df = add_time_features(city_df)
    feat_df = add_lag_features_hourly(feat_df, target)

    # Add cross-feature context if multi-feature data available
    if all_features:
        feat_df = add_cross_feature_context(feat_df, all_features, target)

    feat_df = feat_df.dropna()

    feat_cols = get_xgb_feature_cols(feat_df)
    y = feat_df[target].values.copy()
    use_log = target in LOG_FEATURES
    if use_log:
        y = np.log1p(np.clip(y, 0, None))

    X = feat_df[feat_cols].values

    # Chronological 85/15 split for early stopping
    split_idx = int(len(X) * 0.85)
    X_tr, X_es = X[:split_idx], X[split_idx:]
    y_tr, y_es = y[:split_idx], y[split_idx:]

    model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=XGB_EARLY_STOP)
    model.fit(X_tr, y_tr,
              eval_set=[(X_es, y_es)],
              verbose=False)
    model._arian_log = use_log
    model._arian_best_iter = model.best_iteration
    return model, feat_cols


def predict_xgb_recursive(model, feat_cols: List[str], target: str,
                           history_df: pd.DataFrame,
                           steps: int,
                           cross_feat_means: dict = None) -> np.ndarray:
    """Fast recursive multi-step hourly forecast using numpy buffers.
    Accepts pre-computed cross-feature means for consistent context."""
    use_log = getattr(model, "_arian_log", False)
    h = history_df.sort_values("Timestamp")

    # Prepare value buffer
    vals = h[target].ffill().fillna(0).values.copy().astype(float)
    if use_log:
        vals = np.log1p(np.clip(vals, 0, None))
    buf = list(vals[max(0, len(vals) - _MAX_LOOKBACK):])

    # Pre-compute time features for all future timestamps at once
    last_ts = h["Timestamp"].max()
    future_ts = pd.date_range(last_ts + pd.Timedelta(hours=1),
                              periods=steps, freq="h")
    t_hours    = future_ts.hour.values.astype(float)
    t_dow      = future_ts.dayofweek.values.astype(float)
    t_doy      = future_ts.dayofyear.values.astype(float)
    t_months   = future_ts.month.values.astype(float)
    t_weeks    = np.array(future_ts.isocalendar().week, dtype=float)
    t_hour_sin = np.sin(2 * np.pi * t_hours / 24)
    t_hour_cos = np.cos(2 * np.pi * t_hours / 24)
    t_doy_sin  = np.sin(2 * np.pi * t_doy / 365.25)
    t_doy_cos  = np.cos(2 * np.pi * t_doy / 365.25)
    t_month_sin = np.sin(2 * np.pi * t_months / 12)
    t_month_cos = np.cos(2 * np.pi * t_months / 12)

    # Column-name -> position map
    col_idx = {c: i for i, c in enumerate(feat_cols)}
    n_feat = len(feat_cols)

    preds = np.empty(steps)
    row   = np.zeros((1, n_feat))

    _time_arrays = [("hour", t_hours), ("dayofweek", t_dow),
                    ("dayofyear", t_doy), ("month", t_months), ("week", t_weeks),
                    ("hour_sin", t_hour_sin), ("hour_cos", t_hour_cos),
                    ("doy_sin", t_doy_sin), ("doy_cos", t_doy_cos),
                    ("month_sin", t_month_sin), ("month_cos", t_month_cos)]

    # Pre-fill cross-feature columns (constant during forecast)
    if cross_feat_means:
        for col_name, val in cross_feat_means.items():
            if col_name in col_idx:
                row[0, col_idx[col_name]] = val

    for i in range(steps):
        # ── Time features ─────────────────────────────────────────────
        for name, arr in _time_arrays:
            if name in col_idx:
                row[0, col_idx[name]] = arr[i]

        # ── Lag features ──────────────────────────────────────────────
        n_buf = len(buf)
        for L in _XGB_LAGS:
            key = f"lag_{L}"
            if key in col_idx:
                row[0, col_idx[key]] = buf[-L] if n_buf >= L else 0.0

        # ── Rolling features ──────────────────────────────────────────
        for W in _XGB_ROLLS:
            window = buf[-W:] if n_buf >= W else buf[:]
            w_arr = np.asarray(window)
            w_len = len(w_arr)
            w_mean = float(w_arr.mean())
            for suffix, val in [("mean", w_mean),
                                ("std", float(w_arr.std(ddof=1)) if w_len > 1 else 0.0),
                                ("min", float(w_arr.min())),
                                ("max", float(w_arr.max()))]:
                key = f"roll_{W}_{suffix}"
                if key in col_idx:
                    row[0, col_idx[key]] = val

        # Trend
        if "trend_24_168" in col_idx:
            m24 = float(np.mean(buf[-24:])) if n_buf >= 24 else float(np.mean(buf))
            m168 = float(np.mean(buf[-168:])) if n_buf >= 168 else float(np.mean(buf))
            row[0, col_idx["trend_24_168"]] = m24 - m168

        # ── Predict ───────────────────────────────────────────────────
        yhat_raw = float(model.predict(row)[0])
        if use_log:
            yhat = float(np.expm1(yhat_raw).clip(min=0))
            buf.append(np.log1p(max(yhat, 0.0)))
        else:
            yhat = yhat_raw
            buf.append(yhat)
        preds[i] = yhat

        if len(buf) > _MAX_LOOKBACK + 50:
            buf = buf[-_MAX_LOOKBACK:]

    return preds

print("XGBoost helpers ready  [cross-feature context, early stopping, "
      "trend features, rolling min/max]")

XGBoost helpers ready  [cross-feature context, early stopping, trend features, rolling min/max]


## §6 · Stacking — fit non-negative weights on validation window

In [32]:
def fit_stack_weights(val_actual: np.ndarray,
                      val_preds: Dict[str, np.ndarray]) -> Dict[str, float]:
    keys = list(val_preds.keys())
    P = np.column_stack([val_preds[k] for k in keys])

    def loss(w):
        return np.sqrt(np.mean((P @ w - val_actual) ** 2))

    n = len(keys)
    res = minimize(
        loss, x0=np.ones(n) / n, method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
    )
    return dict(zip(keys, res.x))

print("Stacker ready")

Stacker ready


## §7 · Train & evaluate — per city × feature

For each (city, feature) we:
1. Split train / validation / test chronologically
2. Fit Prophet, SARIMA, XGBoost
3. Fit stacking weights on validation
4. Report stacked test RMSE/MAE

**Note:** For efficiency, XGBoost recursive prediction is capped at the test
window length (168 hours). Prophet and SARIMA are analytical forecasts and
scale easily.

In [ ]:
import time as _time

results: list = []
trained_bundles: dict = {}   # (city, feature) -> model bundle
_t0 = _time.time()

for city in tqdm(sorted(CITIES.keys()), desc="Cities"):
    city_df = (hourly[hourly["City"] == city]
               .sort_values("Timestamp").reset_index(drop=True))
    if len(city_df) < MIN_TRAIN_HOURS:
        print(f"   ⚠  {city}: only {len(city_df)} hours, skipping")
        continue

    # Add derived features once per city
    city_df = add_derived_features(city_df)

    # Identify which features have actual data for this city
    city_features = [f for f in WEATHER_FEATURES if not city_df[f].isna().all()]
    # Also add derived features if they were computed
    derived_feats = [f for f in ("Dew_Point_C", "VPD_kPa") if f in city_df.columns and city_df[f].notna().any()]
    all_predictable = city_features + derived_feats

    for feat in all_predictable:
        if city_df[feat].isna().all():
            continue

        # Chronological split: train | val | test
        test_df  = city_df.iloc[-TEST_HOURS:].copy()
        val_df   = city_df.iloc[-(TEST_HOURS + VAL_HOURS):-TEST_HOURS].copy()
        train_df = city_df.iloc[:-(TEST_HOURS + VAL_HOURS)].copy()

        train_ts  = train_df[feat].ffill().fillna(0)
        train_idx = train_df["Timestamp"]
        val_ts    = val_df[feat].ffill().fillna(0)
        test_ts   = test_df[feat].ffill().fillna(0)

        n_val  = len(val_df)
        n_test = len(test_df)

        # ── Prophet (use ALL available history for accuracy) ───────────
        p_idx = train_idx.copy()
        if hasattr(p_idx.dt, "tz") and p_idx.dt.tz is not None:
            p_idx = p_idx.dt.tz_localize(None)

        try:
            prophet_m = fit_prophet_hourly(train_ts, p_idx, feat)
            val_dates  = pd.date_range(val_df["Timestamp"].iloc[0],
                                       periods=n_val, freq="h")
            test_dates = pd.date_range(test_df["Timestamp"].iloc[0],
                                       periods=n_test, freq="h")
            if val_dates.tz is not None:
                val_dates  = val_dates.tz_localize(None)
            if test_dates.tz is not None:
                test_dates = test_dates.tz_localize(None)
            prop_val  = predict_prophet_hourly(prophet_m, val_dates)
            prop_test = predict_prophet_hourly(prophet_m, test_dates)
        except Exception as e:
            prop_val  = np.full(n_val, train_ts.mean())
            prop_test = np.full(n_test, train_ts.mean())
            prophet_m = None

        # ── SARIMA ─────────────────────────────────────────────────────
        try:
            sarima_m  = fit_sarima_hourly(train_ts, feat)
            sar_full  = predict_sarima_hourly(sarima_m, n_val + n_test)
            sar_val   = sar_full[:n_val]
            sar_test  = sar_full[n_val:n_val + n_test]
        except Exception:
            sar_val  = np.full(n_val, train_ts.mean())
            sar_test = np.full(n_test, train_ts.mean())
            sarima_m = None

        # ── XGBoost (with cross-feature context) ──────────────────────
        try:
            xgb_m, feat_cols = fit_xgb_hourly(
                train_df[["Timestamp", feat] + [f for f in all_predictable if f in train_df.columns]],
                feat, all_features=all_predictable)

            # Pre-compute cross-feature means for forecast context
            cross_means = {}
            for col in feat_cols:
                if col.startswith("x_") and col in train_df.columns:
                    cross_means[col] = float(train_df[col].mean())
                elif col.startswith("x_"):
                    # Compute from raw data
                    pass  # will be 0 in the row

            xgb_val  = predict_xgb_recursive(
                xgb_m, feat_cols, feat,
                train_df[["Timestamp", feat]], n_val, cross_means)

            xgb_test = predict_xgb_recursive(
                xgb_m, feat_cols, feat,
                pd.concat([train_df, val_df])[["Timestamp", feat]], n_test, cross_means)
        except Exception:
            xgb_val  = np.full(n_val, train_ts.mean())
            xgb_test = np.full(n_test, train_ts.mean())
            xgb_m, feat_cols = None, []
            cross_means = {}

        # ── Seasonal climatology ───────────────────────────────────────
        val_ts_dates  = pd.date_range(val_df["Timestamp"].iloc[0],
                                      periods=n_val, freq="h")
        test_ts_dates = pd.date_range(test_df["Timestamp"].iloc[0],
                                      periods=n_test, freq="h")
        seasonal_val  = compute_seasonal_naive(train_df, feat, val_ts_dates)
        seasonal_test = compute_seasonal_naive(
            pd.concat([train_df, val_df]), feat, test_ts_dates)

        # ── Clip individual models to physical bounds ──────────────────
        prop_val  = clip_to_bounds(prop_val,  feat)
        prop_test = clip_to_bounds(prop_test, feat)
        sar_val   = clip_to_bounds(sar_val,   feat)
        sar_test  = clip_to_bounds(sar_test,  feat)
        xgb_val   = clip_to_bounds(xgb_val,   feat)
        xgb_test  = clip_to_bounds(xgb_test,  feat)

        # ── Stack weights (4 models) ──────────────────────────────────
        weights = fit_stack_weights(
            val_ts.values,
            {"prophet": prop_val, "sarima": sar_val,
             "xgboost": xgb_val, "seasonal": seasonal_val})

        stack_test = (weights["prophet"]  * prop_test +
                      weights["sarima"]   * sar_test  +
                      weights["xgboost"]  * xgb_test  +
                      weights["seasonal"] * seasonal_test)
        stack_test = clip_to_bounds(stack_test, feat)

        # ── Record metrics ─────────────────────────────────────────────
        actual = test_ts.values
        for mname, yhat in {"prophet": prop_test, "sarima": sar_test,
                            "xgboost": xgb_test, "seasonal": seasonal_test,
                            "stack": stack_test}.items():
            results.append({
                "City": city, "Feature": feat, "Model": mname,
                "RMSE": float(np.sqrt(mean_squared_error(actual, yhat))),
                "MAE":  float(mean_absolute_error(actual, yhat)),
                "weight": weights.get(mname, np.nan),
            })

        trained_bundles[(city, feat)] = {
            "prophet": prophet_m, "sarima": sarima_m,
            "xgboost": xgb_m, "feat_cols": feat_cols,
            "weights": weights, "cross_means": cross_means,
        }

    n_done = len([k for k in trained_bundles if k[0] == city])
    elapsed = _time.time() - _t0
    print(f"   ✓ {city} — {n_done} features · {elapsed/60:.1f} min elapsed")

results_df = pd.DataFrame(results)
print(f"\n✅ Trained {len(trained_bundles)} (city, feature) bundles "
      f"in {(_time.time()-_t0)/60:.1f} min")
print(f"   Total score rows: {len(results_df)}")
print(f"\n   Avg stacking weights across all bundles:")
w_summary = (results_df.query("Model in ('prophet','sarima','xgboost','seasonal')")
             .groupby("Model")["weight"].mean().round(3))
print(w_summary.to_string())

Cities:   0%|          | 0/16 [00:00<?, ?it/s]

## §8 · Leaderboard

In [ ]:
leaderboard = (results_df
               .groupby(["Feature", "Model"])
               .agg(RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
                    MAE_mean =("MAE",  "mean"), MAE_std =("MAE",  "std"))
               .round(3).reset_index())
print("=== Per-feature leaderboard (5 models: prophet, sarima, xgboost, seasonal, stack) ===")
print(leaderboard.to_string(index=False))

stack_weights_summary = (results_df
                 .query("Model in ('prophet', 'sarima', 'xgboost', 'seasonal')")
                 .groupby(["Feature", "Model"])["weight"]
                 .mean().unstack().round(3))
print("\n=== Avg stacking weights by feature ===")
print(stack_weights_summary)

# Which model wins per feature?
best_per_feat = (leaderboard.query("Model == 'stack'")
                 [["Feature", "RMSE_mean", "MAE_mean"]]
                 .sort_values("RMSE_mean"))
print("\n=== Stack ensemble RMSE by feature (best to worst) ===")
print(best_per_feat.to_string(index=False))

leaderboard.to_csv(OUTPUTS / "phase3_weather_leaderboard.csv", index=False)

=== Per-feature leaderboard ===
         Feature   Model  RMSE_mean  RMSE_std  MAE_mean  MAE_std
Humidity_percent prophet     19.130     7.893    16.840    7.479
Humidity_percent  sarima     15.910     7.562    14.299    7.091
Humidity_percent   stack     17.178     6.591    15.092    6.181
Humidity_percent xgboost     19.557     7.660    16.027    6.456
   Soil_Moisture prophet      0.043     0.018     0.043    0.018
   Soil_Moisture  sarima      0.029     0.019     0.028    0.019
   Soil_Moisture   stack      0.035     0.018     0.034    0.017
   Soil_Moisture xgboost      0.055     0.033     0.050    0.030
     Soil_Temp_C prophet      6.177     1.332     5.440    1.266
     Soil_Temp_C  sarima      3.680     0.806     3.011    0.741
     Soil_Temp_C   stack      5.452     1.141     4.654    1.103
     Soil_Temp_C xgboost      5.480     1.277     4.383    1.151
   Temperature_C prophet      6.571     1.112     5.950    1.069
   Temperature_C  sarima      3.272     0.982     2.748   

## §9 · Generate the 30-day hourly operational forecast

In [ ]:
forecast_frames = []
_t_fc = _time.time()

# Determine all features we trained (original + derived)
all_trained_feats = sorted(set(f for (_, f) in trained_bundles.keys()))
print(f"Trained features: {all_trained_feats}")

for city in tqdm(sorted(CITIES.keys()), desc="30-day hourly FC"):
    city_df = (hourly[hourly["City"] == city]
               .sort_values("Timestamp").reset_index(drop=True))
    if len(city_df) < MIN_TRAIN_HOURS:
        continue

    # Add derived features for context
    city_df = add_derived_features(city_df)

    last_ts = city_df["Timestamp"].max()
    future_dates = pd.date_range(last_ts + pd.Timedelta(hours=1),
                                 periods=FORECAST_HOURS, freq="h")
    city_fc = pd.DataFrame({"Timestamp": future_dates, "City": city})
    lat, lon = CITIES[city]
    city_fc["Latitude"], city_fc["Longitude"] = lat, lon

    # tz-naive dates for Prophet
    future_dates_naive = (future_dates.tz_localize(None)
                          if future_dates.tz is not None else future_dates)

    # Forecast all trained features (original 9 + derived)
    for feat in all_trained_feats:
        bundle = trained_bundles.get((city, feat))
        if bundle is None:
            city_fc[feat] = np.nan
            continue

        weights = bundle["weights"]
        cross_means = bundle.get("cross_means", {})

        # Prophet
        try:
            prop = predict_prophet_hourly(bundle["prophet"], future_dates_naive)
        except Exception:
            prop = np.full(FORECAST_HOURS, city_df[feat].mean() if feat in city_df else 0)

        # SARIMA
        try:
            sar = predict_sarima_hourly(bundle["sarima"], FORECAST_HOURS)
        except Exception:
            sar = np.full(FORECAST_HOURS, city_df[feat].mean() if feat in city_df else 0)

        # XGBoost — recursive (use enough history for _MAX_LOOKBACK)
        try:
            seed = city_df[["Timestamp", feat]].iloc[-500:].copy()
            xgb_fc = predict_xgb_recursive(
                bundle["xgboost"], bundle["feat_cols"], feat,
                seed, FORECAST_HOURS, cross_means)
        except Exception:
            xgb_fc = np.full(FORECAST_HOURS, city_df[feat].mean() if feat in city_df else 0)

        # Seasonal climatology
        seasonal_fc = compute_seasonal_naive(city_df, feat, future_dates)

        # Clip individual models before blending
        prop = clip_to_bounds(prop, feat)
        sar  = clip_to_bounds(sar,  feat)
        xgb_fc = clip_to_bounds(xgb_fc, feat)

        # Horizon-aware blending (XGBoost decays after 72 h)
        stacked = apply_horizon_blend(prop, sar, xgb_fc, seasonal_fc,
                                      weights, FORECAST_HOURS)
        stacked = clip_to_bounds(stacked, feat)
        city_fc[feat] = stacked

    forecast_frames.append(city_fc)

forecast_30d = pd.concat(forecast_frames, ignore_index=True)
forecast_30d.to_parquet(OUTPUTS / "phase3_weather_hourly_30d.parquet", index=False)

print(f"\n✅ 30-day hourly forecast: {len(forecast_30d):,} rows "
      f"({forecast_30d['City'].nunique()} cities x {FORECAST_HOURS} hours) "
      f"in {(_time.time()-_t_fc)/60:.1f} min")

# NaN + bounds check
print("\nFeature forecast quality:")
for feat in all_trained_feats:
    if feat not in forecast_30d.columns:
        continue
    col = forecast_30d[feat]
    nan_pct = col.isna().mean() * 100
    lo, hi = PHYSICAL_BOUNDS.get(feat, (None, None))
    if col.notna().any():
        print(f"   {feat:25s}  NaN {nan_pct:5.1f}%  "
              f"range [{col.min():.2f}, {col.max():.2f}]  "
              f"bounds [{lo}, {hi}]")
    else:
        print(f"   {feat:25s}  ⚠ ALL NaN — no trained bundle")

forecast_30d.head(10)

30-day hourly FC:   0%|          | 0/16 [00:00<?, ?it/s]

✅ 30-day hourly forecast: 11,520 rows (16 cities × 720 hours) in 0.7 min
   Features: ['Temperature_C', 'Humidity_percent', 'Rain_mm', 'Wind_Speed_kmh', 'Wind_Direction_deg', 'Pressure_hPa', 'Solar_Radiation_Wm2', 'Soil_Temp_C', 'Soil_Moisture']

   NaN fraction per feature:
Temperature_C          0.0
Humidity_percent       0.0
Rain_mm                1.0
Wind_Speed_kmh         1.0
Wind_Direction_deg     1.0
Pressure_hPa           1.0
Solar_Radiation_Wm2    1.0
Soil_Temp_C            0.0
Soil_Moisture          0.0


,Timestamp,City,Latitude,Longitude,Temperature_C,Humidity_percent,Rain_mm,Wind_Speed_kmh,Wind_Direction_deg,Pressure_hPa,Solar_Radiation_Wm2,Soil_Temp_C,Soil_Moisture
0,2026-04-28 00:00:00+00:00,Baku,40.4093,49.8671,11.160397,87.375699,NaN,NaN,NaN,NaN,NaN,12.052401,0.255427
1,2026-04-28 01:00:00+00:00,Baku,40.4093,49.8671,11.069088,86.597101,NaN,NaN,NaN,NaN,NaN,12.349844,0.254737
2,2026-04-28 02:00:00+00:00,Baku,40.4093,49.8671,11.029568,85.765651,NaN,NaN,NaN,NaN,NaN,12.702192,0.254696
3,2026-04-28 03:00:00+00:00,Baku,40.4093,49.8671,11.226591,84.630352,NaN,NaN,NaN,NaN,NaN,13.220908,0.254726
4,2026-04-28 04:00:00+00:00,Baku,40.4093,49.8671,11.747802,82.691956,NaN,NaN,NaN,NaN,NaN,14.005845,0.254773


## §10 · Interactive weather map — Plotly dashboard with date selector

The user can select any date in the 30-day window from a dropdown.
For each city, the map shows predicted temperature, humidity, wind,
rain, and other features as a tooltip/popup.

In [ ]:
# ─── Build daily-aggregated version for the map ───────────────────────────
fc = forecast_30d.copy()
fc["Date"] = fc["Timestamp"].dt.date

# Daily aggregates per city for the map view
agg_dict = {}
for feat in WEATHER_FEATURES:
    if feat in fc.columns:
        agg_dict[feat] = "mean"
daily_fc = fc.groupby(["City", "Date"]).agg(agg_dict).reset_index()

# Also add max/min for temperature and rain sum
if "Temperature_C" in fc.columns:
    t_agg = fc.groupby(["City", "Date"])["Temperature_C"].agg(["max", "min"]).reset_index()
    t_agg.columns = ["City", "Date", "Temp_max", "Temp_min"]
    daily_fc = daily_fc.merge(t_agg, on=["City", "Date"], how="left")
if "Rain_mm" in fc.columns:
    r_agg = fc.groupby(["City", "Date"])["Rain_mm"].sum().reset_index()
    r_agg.columns = ["City", "Date", "Rain_total_mm"]
    daily_fc = daily_fc.merge(r_agg, on=["City", "Date"], how="left")
if "Wind_Speed_kmh" in fc.columns:
    w_agg = fc.groupby(["City", "Date"])["Wind_Speed_kmh"].max().reset_index()
    w_agg.columns = ["City", "Date", "Wind_max"]
    daily_fc = daily_fc.merge(w_agg, on=["City", "Date"], how="left")

daily_fc["Latitude"]  = daily_fc["City"].map(lambda c: CITIES[c][0])
daily_fc["Longitude"] = daily_fc["City"].map(lambda c: CITIES[c][1])
dates_sorted = sorted(daily_fc["Date"].unique())

print(f"Daily forecast: {len(daily_fc)} city-days, {len(dates_sorted)} dates")


# ─── Build a rich HTML popup showing ALL measurements ─────────────────────
def _popup_html(row, date):
    """Generate a mini weather-card popup with all features."""
    html = (f"<div style='font-family:Arial,sans-serif;min-width:250px;'>"
            f"<h4 style='margin:0 0 6px 0;border-bottom:2px solid #333;'>"
            f"{row['City']}</h4>"
            f"<span style='color:#777;font-size:11px;'>{date}</span><br><br>")

    # Primary measurements table
    html += "<table style='width:100%;font-size:12px;border-collapse:collapse;'>"
    html += "<tr style='background:#f0f0f0;font-weight:bold;'><td>Measurement</td><td>Avg</td><td>Unit</td></tr>"
    for feat in WEATHER_FEATURES:
        if feat in row.index and pd.notna(row[feat]):
            unit = FEAT_UNITS.get(feat, "")
            name = feat.replace("_", " ").replace(" percent", " %")
            val  = row[feat]
            # Color-code extremes
            style = ""
            if feat == "Temperature_C" and val > 35:
                style = "color:#e74c3c;font-weight:bold;"
            elif feat == "Temperature_C" and val < 0:
                style = "color:#3498db;font-weight:bold;"
            elif feat == "Humidity_percent" and val < 20:
                style = "color:#e67e22;font-weight:bold;"
            elif feat == "Rain_mm" and val > 5:
                style = "color:#2980b9;font-weight:bold;"
            elif feat == "Wind_Speed_kmh" and val > 40:
                style = "color:#e74c3c;font-weight:bold;"
            html += (f"<tr><td>{name}</td>"
                     f"<td style='{style}'>{val:.1f}</td>"
                     f"<td>{unit}</td></tr>")
    html += "</table>"

    # Extra summary
    extras = []
    if "Temp_max" in row.index and pd.notna(row.get("Temp_max")):
        extras.append(f"Temp range: {row['Temp_min']:.1f} - {row['Temp_max']:.1f} C")
    if "Rain_total_mm" in row.index and pd.notna(row.get("Rain_total_mm")):
        extras.append(f"Daily rain total: {row['Rain_total_mm']:.1f} mm")
    if "Wind_max" in row.index and pd.notna(row.get("Wind_max")):
        extras.append(f"Wind gust max: {row['Wind_max']:.1f} km/h")
    if extras:
        html += "<br><span style='font-size:11px;color:#555;'>"
        html += "<br>".join(extras)
        html += "</span>"

    html += "</div>"
    return html


# ─── Folium map with date layers ──────────────────────────────────────────
import folium
from folium.plugins import MarkerCluster

base_map = folium.Map(location=[40.5, 47.5], zoom_start=7,
                      tiles="cartodbpositron", control_scale=True)

for date in dates_sorted:
    fg = folium.FeatureGroup(name=str(date), show=(date == dates_sorted[0]))
    day_data = daily_fc[daily_fc["Date"] == date]

    for _, row in day_data.iterrows():
        popup_html = _popup_html(row, date)
        temp_val = row.get("Temperature_C", 15)
        if pd.isna(temp_val):
            temp_val = 15
        rain_val = row.get("Rain_total_mm", 0)
        if pd.isna(rain_val):
            rain_val = 0
        wind_val = row.get("Wind_Speed_kmh", 0)
        if pd.isna(wind_val):
            wind_val = 0
        humid_val = row.get("Humidity_percent", 50)
        if pd.isna(humid_val):
            humid_val = 50

        # Color by temperature
        if temp_val < 5:      color = "#3498db"
        elif temp_val < 15:   color = "#27ae60"
        elif temp_val < 25:   color = "#f39c12"
        elif temp_val < 35:   color = "#e74c3c"
        else:                 color = "#8e44ad"

        # Radius encodes rain (bigger = more rain), min 7
        radius = max(7, min(20, 7 + rain_val * 0.5))

        # Tooltip shows multi-feature summary (not just temp)
        tooltip_text = (f"{row['City']} | {temp_val:.0f}C  "
                        f"H:{humid_val:.0f}%  "
                        f"R:{rain_val:.1f}mm  "
                        f"W:{wind_val:.0f}km/h")

        folium.CircleMarker(
            location=[row["Latitude"], row["Longitude"]],
            radius=radius, color=color, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=tooltip_text,
        ).add_to(fg)

    fg.add_to(base_map)

folium.LayerControl(collapsed=False).add_to(base_map)

# Legend
legend_html = """
<div style="position:fixed; bottom:20px; left:20px; z-index:1000;
            background:white; padding:10px; border:2px solid #999;
            border-radius:6px; font-size:12px;">
  <b>Predicted Temperature</b><br>
  <span style="color:#3498db">&#9679;</span> &lt;5 C &nbsp;
  <span style="color:#27ae60">&#9679;</span> 5-15 C &nbsp;
  <span style="color:#f39c12">&#9679;</span> 15-25 C &nbsp;
  <span style="color:#e74c3c">&#9679;</span> 25-35 C &nbsp;
  <span style="color:#8e44ad">&#9679;</span> &gt;35 C<br>
  <b>Marker size</b> = daily rainfall<br>
  <i>Click marker for full weather card &middot; Use layer control to select date</i>
</div>
"""
base_map.get_root().html.add_child(folium.Element(legend_html))

base_map.save(str(OUTPUTS / "phase3_weather_map.html"))
print(f"Saved interactive map -> {OUTPUTS / 'phase3_weather_map.html'}")
base_map

Daily forecast: 480 city-days, 30 dates
💾 Saved interactive map → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase3_weather_map.html


In [ ]:
# ─── Plotly multi-feature dashboard: ALL 9 measurements per city ──────────
from plotly.subplots import make_subplots
import plotly.graph_objects as go

n_feats = len(WEATHER_FEATURES)
city_list = sorted(CITIES.keys())

# --- Dashboard 1: Per-feature heatmaps (City x Date) ---
fig_heat = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f.replace("_", " ") for f in WEATHER_FEATURES[:9]],
    vertical_spacing=0.08, horizontal_spacing=0.06,
)

fc_daily = forecast_30d.copy()
fc_daily["Date"] = fc_daily["Timestamp"].dt.date

for idx, feat in enumerate(WEATHER_FEATURES[:9]):
    r, c = divmod(idx, 3)
    pivot = (fc_daily.groupby(["City", "Date"])[feat].mean()
             .reset_index()
             .pivot(index="City", columns="Date", values=feat))
    fig_heat.add_trace(go.Heatmap(
        z=pivot.values,
        x=[str(d) for d in pivot.columns],
        y=pivot.index.tolist(),
        colorscale="RdYlBu_r" if "Temp" in feat else "Viridis",
        showscale=False,
        name=feat,
    ), row=r+1, col=c+1)

fig_heat.update_layout(
    height=900, width=1400,
    title="30-Day Forecast — All Measurements (City x Date heatmaps)",
    template="plotly_white",
)
fig_heat.write_html(str(OUTPUTS / "phase3_all_features_heatmap.html"))
fig_heat.show()

# --- Dashboard 2: Per-city time series for ALL features ---
# Pick 4 representative cities
sample_cities = [city_list[0], city_list[len(city_list)//3],
                 city_list[2*len(city_list)//3], city_list[-1]]

fig_ts = make_subplots(
    rows=n_feats, cols=len(sample_cities),
    subplot_titles=[f"{c} — {f.replace('_',' ')}"
                    for f in WEATHER_FEATURES for c in sample_cities],
    vertical_spacing=0.02, horizontal_spacing=0.04,
    shared_xaxes=True,
)

for fi, feat in enumerate(WEATHER_FEATURES):
    for ci, city in enumerate(sample_cities):
        city_data = forecast_30d[forecast_30d["City"] == city].sort_values("Timestamp")
        fig_ts.add_trace(go.Scatter(
            x=city_data["Timestamp"], y=city_data[feat],
            mode="lines", line=dict(width=1),
            showlegend=False, name=f"{city}-{feat}",
        ), row=fi+1, col=ci+1)

fig_ts.update_layout(
    height=250 * n_feats, width=1400,
    title="30-Day Hourly Forecast — All 9 Measurements (sample cities)",
    template="plotly_white",
)
fig_ts.write_html(str(OUTPUTS / "phase3_all_features_timeseries.html"))
fig_ts.show()

print(f"Saved dashboards:")
print(f"   {OUTPUTS / 'phase3_all_features_heatmap.html'}")
print(f"   {OUTPUTS / 'phase3_all_features_timeseries.html'}")